# Text Embeddings

Text embeddings are a type of representation that allows words, phrases, and even entire documents to be represented as vectors.

In [ ]:
from ai_dojo import setup_plot_style, show

setup_plot_style()

## Embedding in General

We speak of **embedding** whenever we _transform data items to vectors of real numbers_ - that is, we map data into a **continuous vector space**.

The purpose of creating embeddings is to transform these items into a format that machine learning algorithms can efficiently process (vectors), while also capturing and preserving their original meaning or relationships.

Sometimes, the data items are already vectors. However, we want to map them onto a simpler vector space, one with fewer dimensions - this is **dimensionality reduction**.



## Introduction to Text Embeddings

### Word Embedings: What are they good for?

- Technique for representing words as dense vectors in a high-dimensional space
- Captures semantic and syntactic relationships between words
- Words with similar meanings are mapped closer together in the vector space
- Learned from large text corpora, typically using neural networks
- Enables tasks like word similarity, analogy reasoning, and text classification

### Training a Simple Word Embedding Model 

In the following example we are going to train a very simple word embedding model

In [ ]:
from ai_dojo.datasets import read_iliad

In [ ]:
example_text = read_iliad()
show.text(example_text[500000:], limit=2000)

Let's train a simple `Word2Vec` model with the help of `gensim` and `nltk`:

In [ ]:
from gensim.models import Word2Vec
import nltk
from nltk.tokenize import sent_tokenize, word_tokenize


This tokenizer will handle the splitting of the text into words for us, so that we don't have to care about the implementation details and edge cases:

In [ ]:

nltk.download("punkt_tab")

In [ ]:
sentences = sent_tokenize(example_text.lower())  # Tokenize into sentences
tokenized_sentences = [
    word_tokenize(sentence) for sentence in sentences
]  # Tokenize words in each sentence

In [ ]:
sentences[100:110]

In [ ]:
tokenized_sentences[100:110]

Now to the actual model training:

In [ ]:
model = Word2Vec(
    tokenized_sentences,  # Input data: list of tokenized sentences
    vector_size=2,        # Dimensionality of the word vectors
    window=5,             # Maximum distance between the current and predicted word within a sentence
    min_count=1,          # Ignores all words with total frequency lower than this
    workers=4,            # Number of worker threads to train the model
    epochs=20,            # Number of iterations (epochs) over the corpus
)

In [ ]:
model.train(
    tokenized_sentences,  # Training data
    total_examples=model.corpus_count,  # Total number of sentences
    epochs=model.epochs  # Number of iterations (epochs)
)

Let's check the **word vector similarity** for a few examples:

In [ ]:
model.wv.similarity("father", "mother")


In [ ]:
model.wv.similarity("god", "goddess")


In [ ]:
model.wv.similarity("god", "god")


In [ ]:
model.wv.similarity("god", "routed")


Since we have 2-dimensional vectors, we can try visualizing the word vectors.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def plot_word_vectors(model, words):
    # Set up the plot with Seaborn settings
    sns.set_context("talk")
    sns.set_style("whitegrid")
    
    # Gather vectors and labels for the words
    word_vectors = []
    labels = []
    for word in words:
        if word in model.wv.key_to_index:  # Check if the word is in the model's vocabulary
            word_vectors.append(model.wv[word])
            labels.append(word)
        else:
            print(f"'{word}' not found in the model's vocabulary.")
    
    # Check if we have any vectors to plot
    if not word_vectors:
        print("No known words provided for plotting.")
        return

    # Convert list of vectors into 2D array
    word_vectors = np.array(word_vectors)

    # Create a plot
    plt.figure(figsize=(8, 8))
    plot = sns.scatterplot(x=word_vectors[:, 0], y=word_vectors[:, 1], s=30)

    # Annotate each point with its label
    for i, label in enumerate(labels):
        plt.text(word_vectors[i, 0], word_vectors[i, 1], label, horizontalalignment='center', fontsize=8, color='black')

    plt.title("Word Vectors")
    plt.xlabel("Dimension 1")
    plt.ylabel("Dimension 2")
    plt.show()


In [ ]:

# Example usage
# Assuming `model` is your Word2Vec model and `words` is your list of words
words = ["king", "queen", "man", "woman", "god", "goddess", "war", "wrath", "slave", "master", "spear", "sword", "arrow", "horse", "death", "life", "battle", "honor", "shame", "fame", "gift", "love", "friend", "foe", "fire", "water", "warrior"]
plot_word_vectors(model, words)

## Text/Document Embeddings

Text embedding models are trained in a similar way to capture the semantics of an entire segment of text.

### Demo Text Corpus: IMDB Movie Reviews

The following example motivates use cases for text embeddings. We are going to work with a set of movie reviews which have been labelled as "positive" or "negative".

Here is how to download the dataset:

In [ ]:
import tensorflow_datasets

imdb_dataset, dataset_info = tensorflow_datasets.load('imdb_reviews', with_info=True, as_supervised=True)
train_data, test_data = imdb_dataset['train'], imdb_dataset['test']

A small helper function to extract the text from a `tensorflow` tensor to make it readable for us:

In [ ]:
import tensorflow as tf

def get_text(item: tf.Tensor) -> str:
    """
    Decode a TensorFlow tensor into a UTF-8 string.

    Args:
        item (tf.Tensor): A tensor containing text data, typically encoded in UTF-8 bytes.

    Returns:
        str: The decoded string.
    """
    return item.numpy().decode('utf-8')

Let's look at some examples:

In [ ]:
# Iterate through the train dataset and print examples
for example, label in train_data.take(3):  # Just take 5 examples
    show.text(get_text(example))
    print('Label:', 'Positive' if label.numpy() == 1 else 'Negative')
    print("-----------")

To help structure our data, we turn it into a `pandas` dataframe:

In [ ]:
from pandas import DataFrame, Series

In [ ]:
def dataset_to_dataframe(dataset: tf.data.Dataset) -> DataFrame:
    """
    Convert a TensorFlow supervised dataset into a pandas DataFrame.

    Args:
        dataset (tf.data.Dataset): A TensorFlow dataset in a supervised format (text, label).

    Returns:
        pd.DataFrame: A DataFrame with columns 'text' and 'label'.
    """
    # Initialize lists to hold the data
    texts = []
    labels = []
    
    # Iterate over the dataset
    for text_tensor, label_tensor in tensorflow_datasets.as_numpy(dataset):
        texts.append(text_tensor.decode('utf-8'))
        labels.append(int(label_tensor))
    
    # Create a DataFrame
    df = DataFrame({'text': texts, 'label': labels})
    
    return df

In [ ]:
imdb_data_train = dataset_to_dataframe(imdb_dataset["train"])
imdb_data_test = dataset_to_dataframe(imdb_dataset["test"])

In [ ]:
import pandas

In [ ]:
show.dataframe_with_text(imdb_data_train.head(10))

In [ ]:
show.text(imdb_data_train.iloc[0]["text"])

In [ ]:
document_data = imdb_data_train.sample(n=200)

### Transforming Text into Embeddings

**Using an embedding model served by Ollama**

In [ ]:
import ollama

In [ ]:
embedding_model_name = "nomic-embed-text"

In [ ]:
ollama.pull("nomic-embed-text")

In [ ]:
ollama.embeddings(
  model=embedding_model_name,
  prompt='Llamas are members of the camelid family',
)

<div style="border:2px solid gray; padding:10px; width: 95%;">

**🫳 Hands-on: Turn the movie reviews into embedding vectors**

... using an embedding model served by `ollama`.

Your task in the following is to complete a few code snippets so that we can embed the movie reviews.

**Task 1**: _Implement this function according to the documentation and signature._

In [ ]:
import numpy

In [ ]:
def embed(text: str, embedding_model_name: str) -> numpy.ndarray:
    """
    Generate a vector embedding for the given text using the specified model.

    Args:
        text (str): The input text to be embedded.
        embedding_model_name (str): The name of the embedding model to use.

    Returns:
        numpy.ndarray: The embedding vector representing the input text.
    """
    raise NotImplementedError("TODO")
    return vec

Try your implementation on this test case:

In [ ]:
try:
    embed(
        "Alpacas are also members of the camelid family",
        "mxbai-embed-large",
    )
except NotImplementedError as ni:
    print("The function still needs to be implemented")

**Task 2**: _After implementing the function, use it to add an embedding column to the data_

In [ ]:
try:
    document_data["embedding"]
except KeyError:
    print("Error: The embedding column is still missing")

**Can we visualize embeddings?**

The embedding vectors are still high-dimensional, but we can use a good dimensionality reduction algorithm to look at them in 2D space.

<div style="border:2px solid gray; padding:10px; width: 95%;">

**ℹ t-SNE: A Popular Dimensionality Reduction Method**

- **Purpose**: Captures local data structure, often used for visualization
- **Result**: Low-dimensional map with similar data points close together.
- **Limitations**: Computationally intensive, not ideal for large datasets.

We have prepared a function to apply dimensionality reduction to the embeddings, plot them onto a 2D-graph and color the datapoints by their labels:
 

In [ ]:
import seaborn
import numpy as np
import pandas as pd
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt


def visualize_embeddings(
    embeddings, labels,
):
    """
    Visualize embeddings using T-SNE. Display a color bar if the labels are continuous
    and a legend if the labels are categorical (e.g., integers or booleans).

    Args:
    - embeddings (np.ndarray or pd.Series): A numpy array of shape (n_samples, n_features)
      containing the embeddings or a pandas Series where each element is an ndarray.
    - labels (list or np.ndarray): The labels corresponding to each embedding.
    - perplexity (int): The perplexity parameter for T-SNE, which controls the balance between
      local and global aspects of your data. Typically between 5 and 50.
    - n_iter (int): The number of iterations for optimization.
    - random_state (int): Seed for the random number generator for reproducibility.

    Returns:
    - None: Plots the T-SNE embedding.
    """
    # Check if embeddings is a pandas Series and convert if necessary
    if isinstance(embeddings, pd.Series):
        embeddings = np.vstack(embeddings.values)

    # Initialize T-SNE
    tsne = TSNE(
        n_components=2,
        perplexity=30,
    )

    # Perform T-SNE on embeddings
    tsne_results = tsne.fit_transform(embeddings)

    # Plotting
    plt.figure(figsize=(8, 6))  # Adjusted figure size
    if np.issubdtype(labels.dtype, np.integer) or np.issubdtype(labels.dtype, np.bool_):
        # Categorical labels: using a legend
        unique_labels = np.unique(labels)
        colors = plt.cm.coolwarm(
            np.linspace(0, 1, len(unique_labels))
        )  # Generate colors
        for i, label in enumerate(unique_labels):
            idx = labels == label
            plt.scatter(
                tsne_results[idx, 0],
                tsne_results[idx, 1],
                color=colors[i],
                label=str(label),
                s=10,
            )  # Smaller markers
        plt.legend(title="Label")
    else:
        # Continuous labels: using a color bar
        scatter = plt.scatter(
            tsne_results[:, 0],
            tsne_results[:, 1],
            c=labels,
            cmap="viridis",
            alpha=0.6,
            s=10,
        )
        plt.colorbar(scatter)

    plt.title("T-SNE visualization of embeddings")
    plt.xlabel("T-SNE axis 1")
    plt.ylabel("T-SNE axis 2")
    plt.show()

**Task 3**: _Visualize the embeddings and interpret the result._

In [ ]:
try:
    visualize_embeddings(
        embeddings=document_data["embedding"],
        labels=document_data["label"]
    )
except KeyError:
    print("The embedding column is not there yet")

---
_This notebook is licensed under a [Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International (CC BY-NC-SA 4.0)](https://creativecommons.org/licenses/by-nc-sa/4.0/). Copyright © 2024-2026 [Christian Staudt](https://clstaudt.me) and [Katharina Rasch](https://krasch.io)_